# Sequence to Sequence Learning (_Encoder-Decoder_ model using LSTM)

In this series we'll be building a machine learning model to go from one sequence to another, using PyTorch.   
This will be done on **German to English** translations, but the models can be applied to any problem that involves going from one sequence to another, such as summarization, i.e. going from a sequence to a shorter sequence in the same language.

In this first notebook, we'll start simple to understand the general concepts by implementing the model from the [Sequence to Sequence Learning with Neural Networks](https://arxiv.org/abs/1409.3215) paper.

## Introduction

The most common sequence-to-sequence (seq2seq) models are _encoder-decoder_ models, which commonly use a _recurrent neural network_ (RNN) to _encode_ the source (input) sentence into a single vector. In this notebook, we'll refer to this single vector as a **_context vector_**. We can think of the context vector as being **an abstract representation of the entire input sentence**. This vector is then _decoded_ by a second RNN which learns to output the target (output) sentence by generating it one word at a time.


![](assets/seq2seq1.png)

The above image shows an example translation. The input/source sentence, "guten morgen", is passed through the embedding layer (yellow) and then input into the encoder (green). We also append a _start of sequence_ (`<sos>`) and _end of sequence_ (`<eos>`) token to the start and end of sentence, respectively. **At each time-step, the input to the encoder RNN is both the embedding, $e$, of the current word, $e(x_t)$, as well as the hidden state from the previous time-step, $h_{t-1}$, and the encoder RNN outputs a new hidden state $h_t$**. We can think of the hidden state as a vector representation of the sentence so far. The RNN can be represented as a function of both of $e(x_t)$ and $h_{t-1}$:

$$h_t = \text{EncoderRNN}(e(x_t), h_{t-1})$$

We're using the term RNN generally here, it could be any recurrent architecture, such as an _LSTM_ (Long Short-Term Memory) or a _GRU_ (Gated Recurrent Unit).

Here, we have $X = \{x_1, x_2, ..., x_T\}$, where $x_1 = \text{<sos>}, x_2 = \text{guten}$, etc. The initial hidden state, $h_0$, is usually either initialized to zeros or a learned parameter.

Once the final word, $x_T$, has been passed into the RNN via the embedding layer, we use the final hidden state, $h_T$, as the context vector, i.e. $h_T = z$. This is a vector representation of the entire source sentence.

Now we have our context vector, $z$, we can start decoding it to get the output/target sentence, "good morning". Again, we append start and end of sequence tokens to the target sentence. At each time-step, the input to the decoder RNN (blue) is the embedding, $d$, of current word, $d(y_t)$, as well as the hidden state from the previous time-step, $s_{t-1}$, where the initial decoder hidden state, $s_0$, is the context vector, $s_0 = z = h_T$, i.e. the initial decoder hidden state is the final encoder hidden state. Thus, similar to the encoder, we can represent the decoder as:

$$s_t = \text{DecoderRNN}(d(y_t), s_{t-1})$$

Although the input/source embedding layer, $e$, and the output/target embedding layer, $d$, are both shown in yellow in the diagram they are two different embedding layers with their own parameters.

In the decoder, we need to go from the hidden state to an actual word, therefore at each time-step we use $s_t$ to predict (by passing it through a `Linear` layer, shown in purple) what we think is the next word in the sequence, $\hat{y}_t$.

$$\hat{y}_t = f(s_t)$$

The words in the decoder are always generated one after another, with one per time-step. We always use `<sos>` for the first input to the decoder, $y_1$, but for subsequent inputs, $y_{t>1}$, we will sometimes use the actual, ground truth next word in the sequence, $y_t$ and sometimes use the word predicted by our decoder, $\hat{y}_{t-1}$. This is called _teacher forcing_, see a bit more info about it [here](https://machinelearningmastery.com/teacher-forcing-for-recurrent-neural-networks/).

When training/testing our model, we always know how many words are in our target sentence, so we stop generating words once we hit that many. During inference it is common to keep generating words until the model outputs an `<eos>` token or after a certain amount of words have been generated.

Once we have our predicted target sentence, $\hat{Y} = \{ \hat{y}_1, \hat{y}_2, ..., \hat{y}_T \}$, we compare it against our actual target sentence, $Y = \{ y_1, y_2, ..., y_T \}$, to calculate our loss. We then use this loss to update all of the parameters in our model.

## Preparing Data

First up, importing all the necessary libraries. The main ones we'll be using are:

-   [PyTorch](https://pytorch.org/) for creating the models
-   [spaCy](https://spacy.io/) to assist in the tokenization of the data
-   [datasets](https://huggingface.co/docs/datasets/index) to load and manipulate our data
-   [evaluate](https://huggingface.co/docs/evaluate/index) for calculating metrics

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
import tqdm
import evaluate
from collections import Counter

/opt/miniconda3/envs/nlp_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/miniconda3/envs/nlp_env/lib/python3.11/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/opt/miniconda3/envs/nlp_env/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <EB3FF92A-5EB1-3EE8-AF8B-5923C1265422> /opt/miniconda3/envs/nlp_env/lib/python3.11/site-packages/torchvision/image.so
  Reason: tried: '/opt/miniconda3/envs/nlp_env/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/miniconda3/envs/nlp_env/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/miniconda3/envs/nlp_env/lib/python3.11/lib-dynload/../../libjpeg.9.dylib'

We'll set all possible random seeds for deterministic results.

In [2]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

### Dataset

We load the Multi30k dataset — ~30,000 parallel English-German image captions.

In [3]:
dataset = datasets.load_dataset("bentrevett/multi30k")

In [4]:
train_data, valid_data, test_data = (
    dataset["train"],
    dataset["validation"],
    dataset["test"],
)

In [5]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

### Tokenizers

We use spaCy to split sentences into tokens. Load the English and German models first:

```
python -m spacy download en_core_web_sm
python -m spacy download de_core_news_sm
```

In [6]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

Tokenize all examples: lowercase, trim to max length, and wrap with `<sos>` / `<eos>` tokens.

In [7]:
def tokenize_example(example, max_length, lowercase, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    if lowercase:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

In [8]:
max_length = 1000
lowercase = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "max_length": max_length,
    "lowercase": lowercase,
    "sos_token": sos_token,
    "eos_token": eos_token
}

train_data = train_data.map(tokenize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(tokenize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(tokenize_example, fn_kwargs=fn_kwargs)

In [9]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

### Vocabularies

Build a vocabulary from training tokens. Tokens appearing fewer than `min_freq` times are treated as `<unk>`. Special tokens (`<unk>`, `<pad>`, `<sos>`, `<eos>`) are placed at indices 0-3.

In [10]:
class Vocab:
    def __init__(self, tokens_iterator, min_freq=1, specials=None):
        specials = specials or []
        counter = Counter()
        for tokens in tokens_iterator:
            counter.update(tokens)

        self.itos = list(specials) + [
            tok for tok, freq in counter.most_common()
            if freq >= min_freq and tok not in specials
        ]
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        self.default_index = self.stoi[unk_token]

    def __getitem__(self, token):
        return self.stoi.get(token, self.default_index)

    def __contains__(self, token):
        return token in self.stoi

    def __len__(self):
        return len(self.itos)

    def set_default_index(self, index):
        self.default_index = index

    def lookup_tokens(self, indices):
        return [self.itos[i] for i in indices]

    def lookup_indices(self, tokens):
        return [self.stoi.get(token, self.default_index) for token in tokens]

In [11]:
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"

special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

In [12]:
en_vocab = Vocab(train_data["en_tokens"], min_freq=min_freq, specials=special_tokens)
de_vocab = Vocab(train_data["de_tokens"], min_freq=min_freq, specials=special_tokens)

In [13]:
len(en_vocab), len(de_vocab)

(5893, 7853)

In [14]:
en_vocab["the"], en_vocab["The"], en_vocab["<unk>"]      # uses "__getitem__()" under the hood!

(7, 0, 0)

In [15]:
tokens = ["i", "love", "watching", "crime", "shows"]

In [16]:
en_vocab.lookup_indices(tokens)

[951, 2217, 171, 0, 815]

In [17]:
en_vocab.lookup_tokens(range(10))

['<unk>', '<pad>', '<sos>', '<eos>', 'a', '.', 'in', 'the', 'on', 'man']

In [18]:
en_vocab.lookup_tokens([9])

['man']

In [19]:
de_vocab.lookup_tokens(range(10))

['<unk>', '<pad>', '<sos>', '<eos>', '.', 'ein', 'einem', 'in', 'eine', ',']

In [20]:
en_vocab.lookup_indices(["the"])

[7]

In [21]:
"the" in en_vocab           # uses "__contains__()" under the hood!

True

In [22]:
"The" in en_vocab

False

However, here we'll programmatically get it and also check that both our vocabularies have the same index for the unknown and padding tokens as this simplifies some code later on.

We also save the index of our `<unk>` and `<pad>` token, as we'll use it later

In [23]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

And we can get the token corresponding to that index to prove it's the <unk> token.

In [24]:
en_vocab.lookup_tokens([0, 1])

['<unk>', '<pad>']

In [25]:
tokens = ["i", "love", "watching", "crime", "shows"]

In [26]:
en_vocab.lookup_indices(tokens)

[951, 2217, 171, 0, 815]

In [27]:
en_vocab.lookup_tokens(en_vocab.lookup_indices(tokens))

['i', 'love', 'watching', '<unk>', 'shows']

Hopefully we've now got the gist of how the Vocab class works. Time to put it into action!

Just like our tokenize_example, we create a numericalize_example function which we'll use with the map method of our dataset. This will "numericalize" (a fancy way of saying convert tokens to indices) our tokens in each example using the vocabularies and return the result into new "en_ids" and "de_ids" features.

### Numericalize

Neural networks work with numbers, not strings. We convert each token to its corresponding index using the vocabularies we just built.

In [28]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = [en_vocab[tok] for tok in example["en_tokens"]]
    de_ids = [de_vocab[tok] for tok in example["de_tokens"]]
    return {"en_ids": en_ids, "de_ids": de_ids}

Apply the numericalization to all splits using `.map()`. The `example` argument is passed automatically; additional arguments go in `fn_kwargs`.

In [29]:
fn_kwargs = {"en_vocab": en_vocab, "de_vocab": de_vocab}

train_data = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)

Each example now has `en_ids` and `de_ids` — lists of integers. We convert them to PyTorch tensors using `.with_format("torch")` so they're ready for the model.

In [30]:
train_data = train_data.with_format("torch", columns=["en_ids", "de_ids"], output_all_columns=True)
valid_data = valid_data.with_format("torch", columns=["en_ids", "de_ids"], output_all_columns=True)
test_data = test_data.with_format("torch", columns=["en_ids", "de_ids"], output_all_columns=True)

In [31]:
train_data[0]

{'en_ids': tensor([   2,   16,   24,   15,   25,  774,   17,   57,   80,  202, 1305,    5,
            3]),
 'de_ids': tensor([   2,   18,   26,  253,   30,   84,   20,   88,    7,   15,  110, 5374,
         3099,    4,    3]),
 'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

In [32]:
type(train_data[0]["en_ids"])

torch.Tensor

In [33]:
en_vocab.lookup_tokens(train_data[0]["en_ids"])

['<sos>',
 'two',
 'young',
 ',',
 'white',
 'males',
 'are',
 'outside',
 'near',
 'many',
 'bushes',
 '.',
 '<eos>']

### Data Loaders

Sentences have different lengths, but PyTorch batches require tensors of the same shape. We solve this by padding shorter sequences with `<pad>` tokens to match the longest sequence in each batch.

The `collate_fn` handles this padding, and the `DataLoader` wraps our dataset into iterable batches.

In [34]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        return {"en_ids": batch_en_ids, "de_ids": batch_de_ids}
    return collate_fn

In [35]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    return torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )

In [36]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

## Building the Model

We build three components: an **Encoder**, a **Decoder**, and a **Seq2Seq** wrapper that connects them.

### Encoder

The encoder reads the input sentence (German) token by token, embeds each token into a dense vector, and passes it through a multi-layer LSTM. The final hidden and cell states become the **context vector** — a summary of the entire input sentence.

$$
\begin{align*}
(h_t, c_t) &= \text{LSTM}(e(x_t), h_{t-1}, c_{t-1})
\end{align*}
$$

We can just think of $c_t$ as another type of hidden state. Similar to $h_0^l$, $c_0^l$ will be initialized to a tensor of all zeros. Also, our context vector will now be both the final hidden state and the final cell state, i.e. $z^l = (h_T^l, c_T^l)$.

Extending our multi-layer equations to LSTMs, we get:

$$
\begin{align*}
(h_t^1, c_t^1) &= \text{EncoderLSTM}^1(e(x_t), (h_{t-1}^1, c_{t-1}^1))\\
(h_t^2, c_t^2) &= \text{EncoderLSTM}^2(h_t^1, (h_{t-1}^2, c_{t-1}^2))
\end{align*}
$$

> **Note**: Only our hidden state from the first layer is passed as input to the second layer, and not the cell state.

So our encoder looks something like this:

![](assets/seq2seq2.png)

We create this in code by making an `Encoder` module, which requires we inherit from `torch.nn.Module` and use the `super().__init__()` as some boilerplate code. The encoder takes the following arguments:

-   `input_dim` is the size/dimensionality of the one-hot vectors that will be input to the encoder. This is equal to the input (source) vocabulary size.
-   `embedding_dim` is the dimensionality of the embedding layer. This layer converts the one-hot vectors into dense vectors with `embedding_dim` dimensions.
-   `hidden_dim` is the dimensionality of the hidden and cell states.
-   `n_layers` is the number of layers in the RNN.
-   `dropout` is the amount of dropout to use. This is a regularization parameter to prevent overfitting. Check out [this](https://www.coursera.org/lecture/deep-neural-network/understanding-dropout-YaGbR) for more details about dropout.

One thing to note is that the `dropout` argument to the LSTM is how much dropout to apply between the layers of a multi-layer RNN, i.e. between the hidden states output from layer $l$ and those same hidden states being used for the input of layer $l+1$.

In the `forward` method, we pass in the source sentence, $X$, which is converted into dense vectors using the `embedding` layer, and then dropout is applied. These embeddings are then passed into the RNN. As we pass a whole sequence to the RNN, it will automatically do the recurrent calculation of the hidden states over the whole sequence for us! Notice that we do not pass an initial hidden or cell state to the RNN. This is because, as noted in the [documentation](https://pytorch.org/docs/stable/nn.html#torch.nn.LSTM), that if no hidden/cell state is passed to the RNN, it will automatically create an initial hidden/cell state as a tensor of all zeros.

The RNN returns: `outputs` (the top-layer hidden state for each time-step), `hidden` (the final hidden state for each layer, $h_T$, stacked on top of each other) and `cell` (the final cell state for each layer, $c_T$, stacked on top of each other).

As we only need the final hidden and cell states (to make our context vector), `forward` only returns `hidden` and `cell`.

The sizes of each of the tensors is left as comments in the code. In this implementation `n_directions` will always be 1, however note that bidirectional RNNs (covered in tutorial 3) will have `n_directions` as 2.

In [37]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):                             # src      = [src length, batch size]
        embedded = self.dropout(self.embedding(src))    # embedded = [src length, batch size, embedding dim]
        outputs, (hidden, cell) = self.rnn(embedded)    # hidden, cell = [n layers * n directions, batch size, hidden dim]
        return hidden, cell

### Decoder
The decoder generates the output sentence (English) one token at a time. At each step it takes a single token and the context vector (hidden/cell states from the encoder), passes the token through an embedding layer, runs it through an LSTM, and predicts the next token via a linear layer.

Next, we'll build our decoder, which will also be a 2-layer (4 in the paper) LSTM.

![](assets/seq2seq3.png)

The `Decoder` class does a single step of decoding, i.e. it ouputs single token per time-step. The first layer will receive a hidden and cell state from the previous time-step, $(s_{t-1}^1, c_{t-1}^1)$, and feeds it through the LSTM with the current embedded token, $y_t$, to produce a new hidden and cell state, $(s_t^1, c_t^1)$. The subsequent layers will use the hidden state from the layer below ($s_t^{l-1}$) as input, and the previous hidden and cell states from their layer, $(s_{t-1}^l, c_{t-1}^l)$. This provides equations very similar to those in the encoder.

$$
\begin{align*}
(s_t^1, c_t^1) = \text{DecoderLSTM}^1(d(y_t), (s_{t-1}^1, c_{t-1}^1))\\
(s_t^2, c_t^2) = \text{DecoderLSTM}^2(s_t^1, (s_{t-1}^2, c_{t-1}^2))
\end{align*}
$$

Remember that the initial hidden and cell states to our decoder are our context vectors, which are the final hidden and cell states of our encoder from the same layer, i.e. $(s_0^l,c_0^l)=z^l=(h_T^l,c_T^l)$.

We then pass the hidden state from the top layer of the RNN, $s_t^L$, through a linear layer, $f$, to make a prediction of what the next token in the target (output) sequence should be, $\hat{y}_{t+1}$.

$$\hat{y}_{t+1} = f(s_t^L)$$

The arguments and initialization are similar to the `Encoder` class, except we now have an `output_dim` which is the size of the vocabulary for the output/target language. There is also the addition of the `Linear` layer, used to make the predictions from the top layer hidden state.

Within the `forward` method, we accept a batch of input tokens, previous hidden and cell states.   
As we are only decoding one token at a time, the input tokens will always have a sequence length of 1. We `unsqueeze` the input tokens to add a sentence length dimension of 1.  
Then, similar to the encoder, we pass through an embedding layer and apply dropout.  
This batch of embedded tokens is then passed into the RNN with the previous hidden and cell states.  
This produces an `output` (hidden state from the top layer of the RNN), a new `hidden` state (one for each layer, stacked on top of each other) and a new `cell` state (also one per layer, stacked on top of each other).  
We then pass the `output` (after getting rid of the sentence length dimension) through the linear layer to receive our `prediction`. We then return the `prediction`, the new `hidden` state and the new `cell` state.

> **Note**: as we always have a sequence length of 1, we could use `nn.LSTMCell`, instead of `nn.LSTM`, as it is designed to handle a batch of inputs that aren't necessarily in a sequence. `nn.LSTMCell` is just a single cell and `nn.LSTM` is a wrapper around potentially multiple cells. Using the `nn.LSTMCell` in this case would mean we don't have to `unsqueeze` to add a fake sequence length dimension, but we would need one `nn.LSTMCell` per layer in the decoder and to ensure each `nn.LSTMCell` receives the correct initial hidden state from the encoder. All of this makes the code less concise -- hence the decision to stick with the regular `nn.LSTM`.



In [38]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):                           # hidden = [n layers, batch size, hidden dim]
        input = input.unsqueeze(0)                                    # input = [1, batch size]
        embedded = self.dropout(self.embedding(input))                # embedded = [1, batch size, embedding dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))   # output = [1, batch size, hidden dim]
        prediction = self.fc_out(output.squeeze(0))                   # prediction = [batch size, output dim]
        return prediction, hidden, cell

### Seq2Seq

The `Seq2Seq` wrapper connects the encoder and decoder.  
It encodes the source sentence, then feeds the context vector into the decoder step by step.  
Our full model will look like this:
![](assets/seq2seq4.png)

The first thing we do in the `forward` method is to create an `outputs` tensor that will store all of our predictions, $\hat{Y}$.

We then feed the input/source sentence, `src`, into the encoder and receive out final hidden and cell states.

The first input to the decoder is the start of sequence (`<sos>`) token. As our `trg` tensor already has the `<sos>` token appended (all the way back when we tokenized our English sentences) we get our $y_1$ by slicing into it. We know how long our target sentences should be (`trg_length`), so we loop that many times. The last token input into the decoder is the one **before** the `<eos>` token -- the `<eos>` token is never input into the decoder.  
> source\[0]("de_ids"): tensor([   2,   18,   26,  253,   30,   84,   20,   88,    7,   15,  110, 5374,
         3099,    4,    3]),  

> target\[0]("en_ids"): tensor([   2,   16,   24,   15,   25,  774,   17,   57,   80,  202, 1305,    5,
            3]),  

During training, we use **teacher forcing** — with some probability we feed the decoder the actual next token instead of its own prediction. This helps the model learn faster by not compounding errors early in training.


During each iteration of the loop, we:

-   pass the input, previous hidden and previous cell states ($y_t, s_{t-1}, c_{t-1}$) into the decoder
-   receive a prediction (which is also the input for the next iteration, hence represented as $\hat{y}_{t+1}$),  
next hidden state and next cell state ($s_{t}, c_{t}$) from the decoder
-   place our prediction, $\hat{y}_{t+1}$ `(output)` in our tensor of predictions, $\hat{Y}$ `(outputs)`
-   decide if we are going to "teacher force" or not
    -   if we do, the next `input` is the ground-truth next token in the sequence, $y_{t+1}$ `(trg[t])`
    -   if we don't, the next `input` is the predicted next token in the sequence, $\hat{y}_{t+1}$ `(top1)`, which we get by doing an `argmax` over the output tensor

Once we've made all of our predictions, we return our tensor full of predictions, $\hat{Y}$ `(outputs)`.

**Note**: our decoder loop starts at 1, not 0. This means the 0th element of our `outputs` tensor remains all zeros. So our `trg` and `outputs` look something like:

$$
\begin{align*}
\text{trg} = [<sos>, &y_1, y_2, y_3, <eos>]\\
\text{outputs} = [0, &\hat{y}_1, \hat{y}_2, \hat{y}_3, <eos>]
\end{align*}
$$

Later on when we calculate the loss, we cut off the first element of each tensor to get:

$$
\begin{align*}
\text{trg} = [&y_1, y_2, y_3, <eos>]\\
\text{outputs} = [&\hat{y}_1, \hat{y}_2, \hat{y}_3, <eos>]
\end{align*}
$$

In [39]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert encoder.hidden_dim == decoder.hidden_dim, "Hidden dimensions must be equal!"
        assert encoder.n_layers == decoder.n_layers, "Number of layers must be equal!"

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src_length, batch size];           # trg = [trg_length, batch size]; trg
        trg_length, batch_size = trg.shape          # trg_length => length of longest target sequence in the batch
        trg_vocab_size = self.decoder.output_dim    # trg_vocab_size => output_dim
        
        # tensor to store decoder outputs           # outputs = [trg_length, batch_size, trg_vocab_size]
        # The value of outputs at 0th time-step will remain as 0 for all examples in the batch
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        
        # initial hidden state of the decoder comes from the last hidden state of the encoder
        hidden, cell = self.encoder(src)    # hidden = [n layers, batch size, hidden dim]
        
        # first input to the decoder is the <sos> token for the entire batch
        # The decoder gets <sos> first because it needs an initial token before it can predict the first actual English word.
        input = trg[0, :]               # trg[0, :] = [2, 2, 2, ..., 2] ie, index of <sos> across the batch as initial row input
        
        for t in range(1, trg_length):                                  # This starts at 1 because position 0 is <sos>.
            output, hidden, cell = self.decoder(input, hidden, cell)    # hidden = [n layers, batch size, hidden dim]
            outputs[t] = output                                         # outputs[t], output = [batch size, trg_vocab_size]
            
            # choose the decoder’s next input
            teacher_force = random.random() < teacher_forcing_ratio      
            top1 = output.argmax(1)                                   # top1 selects the highest-scoring token per batch example
            input = trg[t] if teacher_force else top1                 # input  = [batch_size]
        return outputs

## Training the Model

Now we have our model implemented, we can begin training it.

### Model Initialization

First, we'll initialize our model. As mentioned before, the input and output dimensions are defined by the size of the vocabulary. The embedding dimesions and dropout for the encoder and decoder can be different, but the number of layers and the size of the hidden/cell states must be the same.

We then define the encoder, decoder and then our Seq2Seq model, which we place on the `device`. The `device` is used to tell PyTorch whether a model or a tensor should be processed on a GPU or CPU. The `torch.cuda.is_available()` function returns `True` if a GPU is detected on our machine. Thus, our model will be placed on the GPU, if we have one.


In [40]:
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)

decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)

model = Seq2Seq(encoder, decoder, device).to(device)

We can also count the number of parameters in our model.

In [41]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Encoder has {count_parameters(encoder):,} trainable parameters")
print(f"Decoder has {count_parameters(decoder):,} trainable parameters")
print(f"Seq2Seq has {count_parameters(model):,} trainable parameters")

Encoder has 5,688,576 trainable parameters
Decoder has 8,209,925 trainable parameters
Seq2Seq has 13,898,501 trainable parameters


### Weight Initialization

Next up is initializing the weights of our model. In the paper they state they initialize all weights from a uniform distribution between -0.08 and +0.08, i.e. $\mathcal{U}(-0.08, 0.08)$.

We initialize weights in PyTorch by creating a function which we `apply` to our model. When using `apply`, the `init_weights` function will be called on every module and sub-module within our model. For each module we loop through all of the parameters and sample them from a uniform distribution with `nn.init.uniform_`.


In [42]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)


model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(7853, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(5893, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (fc_out): Linear(in_features=512, out_features=5893, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

### Optimizer

We define our optimizer, which we use to update our parameters in the training loop. Check out [this](http://ruder.io/optimizing-gradient-descent/) post for information about different optimizers. Here, we'll use Adam.


In [43]:
optimizer = optim.Adam(model.parameters())

### Loss Function

Next, we define our loss function. The `CrossEntropyLoss` function calculates both the log softmax as well as the negative log-likelihood of our predictions.

Our loss function calculates the average loss per token, however by passing the index of the `<pad>` token as the `ignore_index` argument we ignore the loss whenever the target token is a padding token.


In [44]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

### Training Loop

Next, we'll define our training loop.

First, we'll set the model into "training mode" with `model.train()`. This will turn on dropout (and batch normalization, which we aren't using) and then iterate through our data iterator.

As stated before, our decoder loop starts at 1, not 0. This means the 0th element of our `outputs` tensor remains all zeros. So our `trg` and `outputs` look something like:

$$
\begin{align*}
\text{trg} = [<sos>, &y_1, y_2, y_3, <eos>]\\
\text{outputs} = [0, &\hat{y}_1, \hat{y}_2, \hat{y}_3, <eos>]
\end{align*}
$$

Here, when we calculate the loss, we cut off the first element of each tensor to get:

$$
\begin{align*}
\text{trg} = [&y_1, y_2, y_3, <eos>]\\
\text{outputs} = [&\hat{y}_1, \hat{y}_2, \hat{y}_3, <eos>]
\end{align*}
$$

At each iteration:

-   get the source and target sentences from the batch, $X$ and $Y$
-   zero the gradients calculated from the last batch
-   feed the source and target into the model to get the output, $\hat{Y}$
-   as the loss function only works on 2d inputs with 1d targets we need to flatten each of them with `.view`
    -   we slice off the first column of the output and target tensors as mentioned above
-   calculate the gradients with `loss.backward()`
-   clip the gradients to prevent them from exploding (a common issue in RNNs)
-   update the parameters of our model by doing an optimizer step
-   sum the loss value to a running total

Finally, we return the loss that is averaged over all batches.


In [45]:
def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device
):
    model.train()                    # puts the model into training mode. It matters primarily because of dropout in this project.
    epoch_loss = 0
    for i, batch in enumerate(data_loader):             # The DataLoader returns the dictionary created by your collate_fn
        src = batch["de_ids"].to(device)                # src = [src length, batch size]
        trg = batch["en_ids"].to(device)                # .to(device) moves the batch tensors to the same device as the model, such as a GPU.
        
        optimizer.zero_grad()                           # clear old gradients
        output = model(src, trg, teacher_forcing_ratio) # output = [trg length, batch size, trg vocab size]
        
        output_dim = output.shape[-1]
        # Remove <sos> from targets[0] and ignore 0 at predictions[0], hence the output[1:] and trg[1:]
        # view(...) combines the time and batch dimensions
        output = output[1:].view(-1, output_dim)        # output = [(trg length - 1) * batch size, trg vocab size]
        trg = trg[1:].view(-1)                             # trg = [(trg length - 1) * batch size]
        
        loss = criterion(output, trg) # Compute cross-entropy loss
        loss.backward()               # computes gradients of the loss wrt every trainable model parameter
        
        # RNNs/LSTMs can suffer from exploding gradients, especially for long sequences. 
        # Gradient clipping prevents an excessively large update.
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip) 
        
        optimizer.step()
        
        epoch_loss += loss.item()
    return epoch_loss / len(data_loader)          # returns the mean batch loss for the epoch.

### Evaluation Loop

Our evaluation loop is similar to our training loop, however as we aren't updating any parameters we don't need to pass an optimizer or a clip value.

We must remember to set the model to evaluation mode with `model.eval()`. This will turn off dropout (and batch normalization, if used).

We use the `with torch.no_grad()` block to ensure no gradients are calculated within the block. This reduces memory consumption and speeds things up.

The iteration loop is similar (without the parameter updates), however we must ensure we turn teacher forcing off for evaluation. This will cause the model to only use it's own predictions to make further predictions within a sentence, which mirrors how it would be used in deployment.


In [46]:
def evaluate_fn(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            src = batch["de_ids"].to(device)            # src = [src length, batch size]
            trg = batch["en_ids"].to(device)            # trg = [trg length, batch size]

            output = model(src, trg, 0)                 # Run the model with no teacher forcing; 
                                                        # output = [trg length, batch size, trg vocab size]
            output_dim = output.shape[-1]            
            output = output[1:].view(-1, output_dim)    # output = [(trg length - 1) * batch size, trg vocab size]
            trg = trg[1:].view(-1)                      # trg = [(trg length - 1) * batch size]
            
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

### Model Training

We can finally start training our model!

At each epoch, we'll be checking if our model has achieved the best validation loss so far. If it has, we'll update our best validation loss and save the parameters of our model (called `state_dict` in PyTorch). Then, when we come to test our model, we'll use the saved parameters used to achieve the best validation loss.

Only when validation loss improves does the notebook save a checkpoint.
`model.state_dict()` contains the learned values—notably the embedding vectors, encoder LSTM weights, decoder LSTM weights, and final output-layer weights. It does not save the full Python model object.
This selects the model that generalizes best to unseen validation data, rather than blindly selecting the model from the final epoch. Later epochs can overfit: training loss may continue to fall while validation loss rises.


We'll be printing out both the loss and the perplexity at each epoch. It is easier to see a change in perplexity than a change in loss as the numbers are much bigger.

In [47]:
n_epochs = 10
clip = 1.0
teacher_forcing_ratio = 0.5              # roughly half of decoder steps use the correct previous target token.

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

 10%|█████▎                                               | 1/10 [04:01<36:11, 241.32s/it]

	Train Loss:   5.051 | Train PPL: 156.138
	Valid Loss:   5.030 | Valid PPL: 152.882


 20%|██████████▌                                          | 2/10 [08:05<32:22, 242.87s/it]

	Train Loss:   4.460 | Train PPL:  86.500
	Valid Loss:   4.824 | Valid PPL: 124.518


 30%|███████████████▉                                     | 3/10 [12:05<28:11, 241.65s/it]

...

	Train Loss:   3.365 | Train PPL:  28.938
	Valid Loss:   4.016 | Valid PPL:  55.497


 90%|███████████████████████████████████████████████▋     | 9/10 [35:54<03:57, 237.54s/it]

	Train Loss:   3.244 | Train PPL:  25.625
	Valid Loss:   3.940 | Valid PPL:  51.400


100%|████████████████████████████████████████████████████| 10/10 [39:47<00:00, 238.72s/it]

	Train Loss:   3.113 | Train PPL:  22.489
	Valid Loss:   3.833 | Valid PPL:  46.211


## Evaluating the Model

The first thing to do is to test the model's performance on the test set.

We'll load the parameters (`state_dict`) that gave our model the best validation loss and run it on the test set to get our test loss and perplexity.


In [48]:
model.load_state_dict(torch.load("tut1-model.pt"))

test_loss = evaluate_fn(model, test_data_loader, criterion, device)

print(f"| Test Loss: {test_loss:.3f} | Test PPL: {np.exp(test_loss):7.3f} |")

/var/folders/58/kdxpbpcs4p368rfgpx57vt5m0000gn/T/ipykernel_53464/2095565514.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("tut1-model.

| Test Loss: 3.827 | Test PPL:  45.928 |


Pretty similar to the validation performance, which is a good sign. It means we aren't overfitting on the validation set.

You might think it's impossible to overfit on the validation set, but it's not. Every time you tweak your hyperparameters (e.g. optimizer, learning rate, model architecture, weight initialization, etc.) in order to get better results on the validation set, you are slowly overfitting those hyperparameters to your validation set. You can also do this on the test set too! Hence, you should evaluate your model on your test set as few times as possible.

Most papers using neural networks for translation don't give their results in terms of loss and perplexity on the test set, they usually give the [BLEU](https://en.wikipedia.org/wiki/BLEU) score. Unlike loss/perplexity, BLEU is a value between zero and one, where higher is better, and [according to the original BLEU paper](https://aclanthology.org/P02-1040.pdf) it has a high correlation with human judgement.

To get our model's BLEU score on the test set, we first need to use our model to translate every example from our test set, which we do with the `translate_sentence` function below. The function first converts the input `sentence` into `tokens`, optionally lowercases each `token`, and then appends the start and end of sequence tokens, `sos_token` and `eos_token`. It then uses the vocabulary to numericalize the tokens into `ids` and converts these `ids` into a `tensor`, adding a "fake" batch dimension, and then passes the `tensor` through the `encoder` to get the `hidden` and `cell` states. We then perform the decoding, starting with the `sos_token`, converting it into a tensor, passing it through the `decoder`, getting the `predicted_token` our model thinks is most likely to be next in the sequence, which we append to our list of `inputs` to the decoder. If the `predicted_token` is the end of sequence token then we stop decoding, if not we continue the loop, using the `predicted_token` as the next input to the decoder. We keep decoding until the decoder outputs the `eos_token` or we hit `max_output_length` (which we use to avoid the decoder just generating tokens forever). Once we've stopped decoding, we convert out inputs into `tokens` using our vocabulary and return them.


In [49]:
def translate_sentence(sentence, model, en_nlp, de_nlp, en_vocab, de_vocab, lowercase, sos_token, eos_token, device, max_output_length=25):
    model.eval()
    with torch.no_grad():
        
        # 1. Tokenize German source sentence
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
        # print(f'tokens: {tokens}')
        
        if lowercase:
            tokens = [token.lower() for token in tokens]
        
        # 2. Add boundary tokens
        tokens = [sos_token] + tokens + [eos_token]
        # print(f'After adding SOS, EOS: {tokens}')

        # 3. Convert German tokens to vocabulary IDs
        ids = de_vocab.lookup_indices(tokens)

        # 4. Create a batch with one sentence
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
        # print(f'Numericalized tensor: {tensor} tensor.shape: {tensor.shape}')

        # 5. Encode the German sentence
        hidden, cell = model.encoder(tensor)

        # 6. Begin decoding from <sos>
        inputs = en_vocab.lookup_indices([sos_token])
        # print('====================')

        # 7. Generate one English token at a time
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            # print(f'inputs_tensor: {inputs_tensor}')
            
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)
            
            predicted_token = output.argmax(-1).item()
            # print(f'predcted_token: {predicted_token}')
            
            inputs.append(predicted_token)
            # print('====================')
            
            if predicted_token == en_vocab[eos_token]:
                break
        
        # 8. Convert predicted IDs to English tokens
        tokens = en_vocab.lookup_tokens(inputs)
        # print(f'tokens: {tokens}')
    return tokens

We'll pass a test example (something the model hasn't been trained on) to use as a sentence to test our `translate_sentence` function, passing in the German sentence and expecting to get something that looks like the English sentence.


In [50]:
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

('Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.',
 'A man in an orange hat starring at something.')

In [51]:
translation = translate_sentence(sentence, model, en_nlp, de_nlp, en_vocab, de_vocab, lowercase, sos_token, eos_token, device,)
print(translation)

['<sos>', 'a', 'man', 'in', 'a', 'hat', 'hat', 'is', 'his', 'his', '.', '.', '<eos>']


Our model has seemed to have figured out that the input sentence mentions a man wearing an item of clothing (though gets both the color and the item wrong), but it can't seem to figure out what the man is doing.

We shouldn't be expecting amazing results, our model is relatively small to what is used in the paper we're implementing (they use four layers with embedding and hidden dimensions of 1000) and is miniscule compared to modern translation models (which have billions of parameters).


The model doesn't just translate examples in the training, validation and test sets. We can use it to translate arbitrary sentences by passing any string to the `translate_sentence`.

Note that the multi30k dataset consists of image captions that have been translated from English to German, and our model has been trained to translate German to English. Therefore, the model will only output reasonable translations if the sentences are German sentences that could potentially be image captions. (It's also important to re-iterate that the model trained here is relatively small and the translation performance will generally be poor.)

Below, we input the German translation of **"A man is watching a film."**


In [52]:
sentence = "Ein Mann sitzt auf einer Bank."

In [53]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lowercase,
    sos_token,
    eos_token,
    device,
)
print(translation)

['<sos>', 'a', 'man', 'sits', 'on', 'a', 'bench', '.', '<eos>']


And we receive our translation, which is reasonably close.

We can now loop over our `test_data`, getting our model's translation of each test sentence.


In [54]:
translations = [
    translate_sentence(example["de"], model, en_nlp, de_nlp, en_vocab, de_vocab, lowercase, sos_token, eos_token, device,)
    for example in tqdm.tqdm(test_data)
]

100%|█████████████████████████████████████████████████| 1000/1000 [00:10<00:00, 94.29it/s]


To calculate BLEU, we'll use the `evaluate` library. It's recommended to use libraries for measuring metrics to ensure there are now bugs in your metric calculations and giving you potentially incorrect results.

The BLEU metric can be loaded from the `evaluate` library like so:


In [55]:
bleu = evaluate.load("bleu")

One quirk one the BLEU metric is that it expects the predictions (predicted translations) to be strings and the references (actual English sentences) to be a list of sentences. This is because BLEU works if you have multiple correct sentences per prediction as there may be potentially be multiple ways to translate a sentence. In our case, we only have a single reference sentence so we just wrap our target sentence in a list. We also convert our translations from a list of tokens into a string by joining them with whitespace inbetween and getting rid of the `<sos>` and `<eos>` tokens (as they will never appear in our reference sentences).


In [56]:
predictions = [" ".join(translation[1:-1]) for translation in translations]

references = [[example["en"]] for example in test_data]

In [57]:
predictions[0], references[0]

('a man in a hat hat is his his . .',
 ['A man in an orange hat starring at something.'])

We also need to define a function which tokenizes an input string. This will be used to calculate the BLEU score by comparing our predicted tokens against the reference tokens.

It seems a bit odd that we joined our translated tokens together into a string only to just tokenize them again, and also used the English string from our test data instead of the existing tokens (`en_tokens`), however this is another quirk of the BLEU metric provided by the `evaluate` library; the `predictions` and `references` must be strings and not tokens, and that we must tell the metric how these strings should be tokenized.

The `get_tokenize_fn` returns our `tokenizer_fn`, which uses our `spaCy` tokenizer and lowercases tokens if necessary.


In [58]:
def get_tokenizer_fn(nlp, lowercase):
    def tokenizer_fn(s):
        tokens = [token.text for token in nlp.tokenizer(s)]
        if lowercase:
            tokens = [token.lower() for token in tokens]
        return tokens

    return tokenizer_fn

In [59]:
tokenizer_fn = get_tokenizer_fn(en_nlp, lowercase)

In [60]:
tokenizer_fn(predictions[0]), tokenizer_fn(references[0][0])

(['a', 'man', 'in', 'a', 'hat', 'hat', 'is', 'his', 'his', '.', '.'],
 ['a', 'man', 'in', 'an', 'orange', 'hat', 'starring', 'at', 'something', '.'])

Finally, we calculate the BLEU metric across our test set!

We pass our `predictions`, `references` and our `tokenizer_fn` to the `compute` method of the BLEU metric to get our results.


In [61]:
results = bleu.compute(
    predictions=predictions, references=references, tokenizer=tokenizer_fn
)

results

{'bleu': 0.12247950394235277,
 'precisions': [0.4615263571990559,
  0.17455166524338173,
  0.08179271708683473,
  0.03810504634397528],
 'brevity_penalty': 0.9729914192096943,
 'length_ratio': 0.973349670699954,
 'translation_length': 12710,
 'reference_length': 13058}

We get a BLEU score of 0.12, not bad for our first translation model.

In the subsequent notebooks we'll be implementing more translation papers and slowly increasing the BLEU score achieved.
